# Alley flag: how the City names alleys, and the frozen rule

**Purpose:** decide how to identify alleys before any model is run (QA/QC item M4).
**Reads:** `data/processed/segments_clean.csv` · **Writes:** nothing. The flag itself is built in
`clean_segments.ipynb` (Cell 4).

In [1]:
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
cl = pd.read_csv(ROOT / "data/processed/segments_clean.csv")
name = cl["rd20full"].astype("string").str.upper().str.strip()

# 1) The last word of every street name (ST, AVE, DR...): shows how the City labels street types
print(name.str.split().str[-1].value_counts().head(25))

# 2) Names that contain ALLEY or ALY as a whole word
m = name.str.contains(r"\bALLEY\b|\bALY\b", regex=True, na=False)
print("alley-like names:", int(m.sum()))
print(name[m].value_counts().head(15))

rd20full
ST           9048
AV           4661
DR           4387
RD           2421
ALLEY        2165
WY           1536
CT           1522
PL           1084
BL           1083
LN            892
TR            212
CR            143
PY            114
RA            109
PT             99
CV             65
STRANDWAY      58
BROADWAY       47
SUR            42
HY             42
GLEN           22
REAL           19
RUIZ           19
MONTENERO      18
VERANO         17
Name: count, dtype: int64
alley-like names: 2173
rd20full
ALLEY      2165
ALLEY E       5
ALLEY C       1
ALLEY H       1
ALLEY A       1
Name: count, dtype: Int64


In [2]:
cl["is_alley"] = name.str.contains(r"\bALLEY\b", regex=True, na=False)
last = name.str.split().str[-1]
print("whole-word ALLEY:", int(cl["is_alley"].sum()), "| ends in ALLEY:", int((last == "ALLEY").sum()))

# Alley-flagged names that DON'T end in ALLEY (check that each one is really an alley)
print(name[cl["is_alley"] & (last != "ALLEY")].value_counts().head(20))

# Clean pool: alleys vs everything else
a = cl[cl["in_clean_set"] == 1]
print(a.groupby("is_alley").agg(n=("pci23", "size"), pci_median=("pci23", "median"),
      width_median=("pwidth", "median"), lanes_median=("lane_no", "median"),
      share_residential=("is_residential", "mean")).round(2))

# Clean alleys vs streets in our 5 driving areas
ours = a["cpname"].str.contains("Mira Mesa|Pacific Beach|North Park|Golden Hill|Southeastern", case=False, na=False)
print(a[ours].groupby(["cpname", "is_alley"]).size().unstack(fill_value=0))

whole-word ALLEY: 2173 | ends in ALLEY: 2165
rd20full
ALLEY E    5
ALLEY C    1
ALLEY H    1
ALLEY A    1
Name: count, dtype: Int64
              n  pci_median  width_median  lanes_median  share_residential
is_alley                                                                  
False     20908       75.40          36.0           2.0                0.9
True       2120       42.55          30.0           2.0                1.0
is_alley                False  True 
cpname                              
GREATER GOLDEN HILL       264     59
MIRA MESA                1287      2
NORTH PARK                615    260
PACIFIC BEACH             674    261
SOUTHEASTERN SAN DIEGO    724    217


### Checkpoint — alley rule (frozen 2026-11-13, before any model was run)

- The City labels alleys with the word "ALLEY" in the street name (`rd20full`): 2,165 names end in it,
  plus 8 lettered alleys ("ALLEY E" ×5, "ALLEY A", "ALLEY C", "ALLEY H"), for 2,173 in all. No
  abbreviation such as "ALY" appears.
- In the clean pool, 2,120 segments are alleys. Their median PCI is 42.55 vs. 75.40 for all other clean
  segments, while their median width (30 vs. 36 ft) and lane count (2 vs. 2) barely differ, and every
  alley passes the two-lane residential filter. Width and lanes therefore cannot identify alleys.
- **Rule:** `is_alley` = the street name contains the whole word ALLEY.
- **Decision:** alleys are excluded from every analysis set. The primary set is clean, residential,
  non-alley (pre-registered 2026-11-12). Otherwise a model could score well by recognizing alleys
  instead of reading the pavement.
- Alleys are common where we drive (clean segments: North Park 260, Pacific Beach 261, Southeastern
  San Diego 217, Greater Golden Hill 59, Mira Mesa 2), so drives avoid them from now on.